# Hypre "Subnormal gamma" issue on `mat_twist` — diagnosis & fix

*2026-05-21*

This notebook documents how we tracked down (and worked around) Hypre's `Subnormal gamma value in PCG` error that fires across the entire `mat_twist` benchmark.

## Part 1 — Problem description

### How the data is produced

PolyFEM runs a `mat_twist` Newton simulation and dumps every Newton iteration's stiffness matrix `A` and RHS `b` to `(step)_(iter)_A.bin` / `(step)_(iter)_b.bin`. PolyFEM itself uses Hypre (BoomerAMG + PCG) as its linear solver.

Our benchmark driver (`TestMatLogger`) replays those matrices through each solver — AMGCL, Hypre, Trilinos, Pardiso — for timing comparison.

### What we observed

From simulation **step ~30 onward**, every iteration's matrix triggers Hypre's `Subnormal gamma value in PCG` error at `mpirun -np 8`. PCG quits after 1–6 steps with residual stuck around `O(10⁻³)`, far above `tol = 1e-10`. The other three solvers handle the same matrices fine.

### The bail-out site

Hypre's `pcg.c` line 707–712:

```c
if (! (gamma > HYPRE_REAL_MIN) )
{
   hypre_error_w_msg(HYPRE_ERROR_CONV, "Subnormal gamma value in PCG");

   break;
}
```

`gamma = (r, M⁻¹r)` is PCG's preconditioned inner product. `HYPRE_REAL_MIN ≈ 2.225 × 10⁻³⁰⁸` is the smallest *normal* IEEE 754 double; once `gamma` slides into the subnormal range the mantissa truncates bit-by-bit and the next arithmetic op typically lands at literal `0` or `inf`, so Hypre bails (`HYPRE_ERROR_CONV = 256`).

## Part 2 — Pinpointing the cause

### Why it matters

PolyFEM solved these same matrices in-process and finished the simulation; the benchmark fails on them. The interesting question is what the benchmark is doing differently, since both call Hypre on the same `A` and `b`.

### The one difference: how the matrix is split across cores

- **PolyFEM**: single process, `--max_threads 16` (OpenMP). Hypre sees `mpi_size = 1`, owns the whole matrix on one rank, runs the *serial* BoomerAMG coarsening and *serial* PCG inner products.
- **Benchmark**: `mpirun -np 8 ./TestMatLogger`. Hypre's `mpi_size = 8`; the IJ matrix is split into 8 contiguous row blocks (our `partition_mode_ = RowBlock` code in `HypreSolver.cpp:96-104`), and BoomerAMG + PCG run in parallel mode with the resulting `ParCSR` data structure.

Same Hypre source, same compile flags, same `A`, same `b`, same default solver parameters. The only difference is **the partition imposed by switching from 1 rank to 8 ranks**.

### Verification A — put all rows on rank 0, keep np=8

If the partition is the cause, then forcing all rows onto rank 0 (other ranks own the empty range `[N, N-1]`) while still running with `mpirun -np 8` should make the subnormal go away. It does:

| partition mode | np | step 71_1 num_iter | final_res_norm | subnormal |
|---|---:|---:|---:|---:|
| `RowBlock` (default) | 8 | 5 | 1.35 × 10⁻³ | 8 ❌ |
| **`RankZero`** (all rows on rank 0) | 8 | **544** | **9.31 × 10⁻¹¹** ✅ | **0** |

Same matrix, same `mpirun -np 8`, just a different partition — and the failure disappears. This rules out "Hypre's MPI code path is broken" and points at the row-block partition itself.

### Verification B — sweep np with the default row-block partition

If row-block partitioning is the cause, the failure should track `mpi_size`. Sweeping np on the same hard matrix (`step 71_1`), default settings:

| np | num_iter | final_res_norm | subnormal |
|---:|---:|---:|---:|
| 1 | 544 | 9.31 × 10⁻¹¹ ✅ | 0 |
| 2 | 573 | 9.63 × 10⁻¹¹ ✅ | 0 |
| **4** | **5** | **1.03 × 10⁻³** ❌ | **4** |
| 8 | 5 | 1.35 × 10⁻³ ❌ | 8 |
| 16 | 5 | 1.29 × 10⁻³ ❌ | 16 |

A clean cliff at **np = 2 → 3 (we verified np=3 already fails)**: as soon as the matrix is sliced into ≥3 row blocks, the row-block partition fragments the matrix's graph enough that BoomerAMG's coarsening can't build a strong-enough preconditioner, `(r, M⁻¹r)` collapses, and PCG hits the subnormal check within a handful of iterations.

### Conclusion of Part 2

The bug is **our naive row-block partition**, not Hypre itself. The partition cuts arbitrary slices of row IDs without regard to the matrix's connectivity, breaking strong connections that BoomerAMG relies on. PolyFEM never hit this because PolyFEM runs Hypre with `mpi_size = 1`.

Parts 3+ work through what to do about it: better partitions (METIS), changing the Krylov method (GMRES) so it doesn't hit the subnormal check, and changing the preconditioner (Euclid ILU) so the partition matters less.

## Interlude — Sanity checks: ruling out "this is just a numerical quirk"

Before chasing config fixes we did three independent sanity checks to confirm the subnormal is **deterministic, structural, and not** caused by a poorly-set numerical option that could be tuned away in isolation. All on `mat_twist / 3146 / 71_1` at `np=8`.

### Check 1 — determinism

Ran the same `mpirun -np 8` solve **five times back-to-back**:

```
run #1   num_iter=5   residual=0.32058   final_res=0.0013501   subnormal=8
run #2   num_iter=5   residual=0.32058   final_res=0.0013501   subnormal=8
run #3   num_iter=5   residual=0.32058   final_res=0.0013501   subnormal=8
run #4   num_iter=5   residual=0.32058   final_res=0.0013501   subnormal=8
run #5   num_iter=5   residual=0.32058   final_res=0.0013501   subnormal=8
```

**Bit-for-bit identical.** This rules out OS scheduling jitter, MPI process binding randomness, and HMIS coarsening's internal random tie-breaking as the cause. The failure is a deterministic function of the inputs.

### Check 2 — MPI_Allreduce algorithm

PCG's inner product `(r, M⁻¹r)` is computed via `MPI_Allreduce`, and different OpenMPI collective algorithms sum the partial results in different orders, which produces *different* floating-point ULPs. If subnormal was being triggered by a particular reduction order, swapping the algorithm should change the iteration count. We forced six OpenMPI tuned algorithms (`--mca coll_tuned_use_dynamic_rules 1 --mca coll_tuned_allreduce_algorithm {1..6}`) — basic_linear, nonoverlapping, recursive_doubling, ring, segmented_ring, rabenseifner:

```
allreduce_algorithm=1   num_iter=5   residual=0.32058   subnormal=8
allreduce_algorithm=2   num_iter=5   residual=0.32058   subnormal=8
allreduce_algorithm=3   num_iter=5   residual=0.32058   subnormal=8
allreduce_algorithm=4   num_iter=5   residual=0.32058   subnormal=8
allreduce_algorithm=5   num_iter=5   residual=0.32058   subnormal=8
allreduce_algorithm=6   num_iter=5   residual=0.32058   subnormal=8
```

Identical across all six. **Reduction ordering is not the trigger.**

### Check 3 — BoomerAMG coarsening type

Maybe the default coarsener (HMIS) is unusually fragile and a different one would dodge subnormal. We swept the four parallel coarseners — CLJP (0), Falgout (6), PMIS (8), HMIS (10) — at `np=8`:

| coarsen_type | num_iter (np=8) | final_res | subnormal |
|---|---:|---:|---:|
| CLJP (0) | 3 | 1.8e-1 | 8 ❌ |
| Falgout (6) | 4 | 3.4e-1 | 8 ❌ |
| PMIS (8) | 5 | 2.8e-1 | 8 ❌ |
| HMIS (10, default) | 5 | 3.2e-1 | 8 ❌ |

All four subnormal-bail within 3–5 iterations. The exact iteration count differs but the failure mode is universal — **switching the coarsening algorithm doesn't help.**

### Why we documented these first

Each of the three checks targets a "soft" cause that would let us patch the issue without rethinking the partition:

- random / timing-noise → would fix itself or be soothed with NUMA pinning,
- floating-point reduction ordering → fixable with `MPI_Allreduce` algorithm flags,
- weak coarsener → fixable by setting a different `CoarsenType` constant in `HypreSolver.cpp`.

All three are ruled out. The remaining structural variable is **how the matrix rows are distributed across ranks** — which is Part 2's conclusion, just arrived at by elimination here.

## Part 3 — Axis sweep

Now that we know the row-block partition is the trigger, the natural question is:
*which of the three independent axes of the Hypre wrapper — partition, Krylov, preconditioner — actually fix the failure, and what does each combination cost?*

We sweep all 2³ = 8 on/off combinations of changing each axis from the
default (`row_block + pcg + boomeramg`) and run the canonical hard matrix
`mat_twist / 3146 / 71_1` at `np=1` and `np=8`. Tolerance is `1e-10`,
`max_iter = 5000` (set in `linear-solver-spec.json`).

The first three columns flag which axes changed vs the baseline.

### Results

| Δ part | Δ krylov | Δ precond | config | np=1 iter | np=1 final_res | np=1 wall | verdict | np=8 iter | np=8 final_res | np=8 wall | verdict |
|:-:|:-:|:-:|---|---:|---:|---:|:-:|---:|---:|---:|:-:|
|  |  |  | row_block + pcg + boomeramg (baseline) | 544 | 9.31e-11 | 1.10 s | ✅ | 5 | 1.35e-3 | 0.05 s | ❌ subn |
| ✓ |  |  | metis + pcg + boomeramg | 544 | 9.31e-11 | 1.11 s | ✅ | 12 | 7.59e-4 | 0.05 s | ❌ subn |
|  | ✓ |  | row_block + gmres + boomeramg | 1079 | 9.98e-11 | 2.39 s | ✅ | 5000 | 2.10e-5 | 4.50 s | ⚠️ stuck |
|  |  | ✓ | row_block + pcg + euclid | 1 | 1.51e-3 | 0.15 s | ❌ subn | **2103** | **8.11e-11** | **0.36 s** | ✅ |
| ✓ | ✓ |  | metis + gmres + boomeramg | 1079 | 9.98e-11 | 2.39 s | ✅ | **1521** | **9.86e-11** | **0.91 s** | ✅ |
| ✓ |  | ✓ | metis + pcg + euclid | 1 | 1.51e-3 | 0.14 s | ❌ subn | 1 | 1.31e-3 | 0.05 s | ❌ subn |
|  | ✓ | ✓ | row_block + gmres + euclid | 5000 | 1.37e-5 | 7.57 s | ⚠️ stuck | 5000 | 6.73e-10 | 1.18 s | ⚠️ near |
| ✓ | ✓ | ✓ | metis + gmres + euclid | 5000 | 1.37e-5 | 7.64 s | ⚠️ stuck | 5000 | 1.42e-5 | 1.24 s | ⚠️ stuck |

(✅ converged to 1e-10; ⚠️ ran to `max_iter` without converging; ❌ subn = Hypre's "Subnormal gamma" bail-out.)

### Reading the table

**`np=1`**: only the four "no precond change" rows converge cleanly (544 iter on baseline / metis no-op, 1079 iter on gmres variants). Every config that switches in Euclid subnormal-bails at `np=1` — Euclid in BJ=1 mode degenerates to a full ILU on the whole near-singular matrix, which is itself unstable. For serial work, **PCG + BoomerAMG is the only stable choice**.

**`np=8`** — the failure mode we actually care about:

- Adding **METIS partition alone** doesn't help — PCG still hits the subnormal check (12 iter vs 5).
- Adding **GMRES alone** ducks the subnormal check but the partition-weakened BoomerAMG can't reach `1e-10` even with 5 000 iter.
- Adding **Euclid alone** is the single biggest win: 2 103 iterations, **0.36 s wall time — 3× faster than the serial baseline**.
- Adding **METIS + GMRES** also converges, in 1 521 iterations and 0.91 s. Slower than Euclid-alone but a genuine SPD-clean stack (no ILU hacks).
- Adding **Euclid + anything-else (METIS, GMRES)** breaks the gain: METIS scrambles the BJ block structure (back to subnormal), and adding GMRES to Euclid forces orthogonalization overhead without enough preconditioning to recover.

### Conclusion of Part 3

Three observations:

1. **Single-axis Euclid (`row_block + pcg + euclid`) is the cheapest fix and the fastest config at `np=8`.** The naive partition is fine when the preconditioner is ILU-based, because ILU's robustness doesn't depend on the matrix's graph structure the way classical AMG does.
2. **Single-axis METIS or GMRES doesn't suffice at `np=8`** — METIS doesn't avoid the subnormal trap, GMRES avoids the trap but inherits the weak preconditioner.
3. **More axes ≠ better.** Two of the three triple-change configurations (`metis+pcg+euclid`, `metis+gmres+euclid`) are worse than the single-axis Euclid config. Mixing METIS with Euclid is actively counter-productive: METIS makes each rank's local block more spatially coherent, which makes its full-block ILU unstable, which makes BJ Euclid fail.

The recommendation: **use `row_block + pcg + euclid` at `np ≥ 2`, keep `row_block + pcg + boomeramg` at `np = 1`**.

### Reference: AMGCL and Trilinos on the same matrix

For context, the same `mat_twist / 3146 / 71_1` matrix run through the other two iterative solvers in the benchmark (`tol = 1e-10`, each solver's own default `max_iter`):

| solver | parallelism | num_iter | final_res | wall | per-iter | verdict |
|---|---|---:|---:|---:|---:|:-:|
| AMGCL (SA-AMG + CG) | OMP=1 | 127 | 9.48e-11 | 2.22 s | 17.5 ms | ✅ |
| AMGCL (SA-AMG + CG) | OMP=8 | 127 | 9.51e-11 | 1.98 s | 15.6 ms | ✅ |
| **Trilinos (MueLu + BlockGMRES) — Trilinos default in `TrilinosSolver.cpp`** | mpi np=1 | 535 | 9.99e-11 | 1.75 s | 3.27 ms | ✅ |
| **Trilinos (MueLu + BlockGMRES) — Trilinos default in `TrilinosSolver.cpp`** | mpi np=8 | 527 | 9.90e-11 | **0.93 s** | 1.76 ms | ✅ |
| Trilinos (MueLu + PseudoBlockCG) | mpi np=1 | 550 | 9.53e-11 | **0.67 s** | 1.22 ms | ✅ |
| Trilinos (MueLu + PseudoBlockCG) | mpi np=8 | 541 | 9.59e-11 | 0.83 s | 1.53 ms | ✅ |

The two **bold** rows are what `TrilinosSolver.cpp:290` instantiates today (`Belos::BlockGmresSolMgr` + MueLu SA-AMG). The PseudoBlockCG rows come from setting `POLYSOLVE_TRILINOS_KRYLOV=cg`, which Phase 8 added — visibly faster for this SPD problem (`np=1`: 0.67 s vs 1.75 s).

### Comparison summary

Setting Hypre's best config (`row_block + pcg + euclid` at `np=8`, **0.36 s / 2103 iter**) next to the references:

- **Fastest overall on this matrix is Hypre + Euclid** at 0.36 s — about 2× faster than Trilinos CG and ~6× faster than AMGCL. The win comes from cheap per-iteration cost (BJ ILU triangular solve, no AMG hierarchy), even though it takes ~4× more iterations than Trilinos.
- **AMGCL** takes the fewest iterations (SA-AMG is genuinely a stronger preconditioner for this elasticity-like operator), but each iteration is expensive (~17 ms — full SA-AMG V-cycle).
- **Trilinos PseudoBlockCG @ np=1** is the most balanced — 1.22 ms/iter and competitive wall time, with the cleanest mathematical pedigree (SPD-correct CG on SA-AMG-preconditioned operator).

So the picture for `mat_twist`-style hard matrices is:

- **Hypre** is competitive when configured correctly (`+ euclid`); the default Hypre config simply fails.
- **AMGCL** and **Trilinos** never see the subnormal issue because (a) they use SA-AMG, which doesn't suffer from the same parallel-coarsening weakening, and (b) their CG/GMRES implementations don't have Hypre's specific `gamma > REAL_MIN` bail-out clause.

## Part 4 — Open questions

The experiments and the SPD discussion leave three decisions for us to make about the benchmark setup. These are the things worth aligning on before locking the configuration.

### Q1. Hypre default — which stack, and should it be `np`-dependent?

**Context.** The current default is `row_block + pcg + boomeramg`, which is what triggers the subnormal failure at `np ≥ 3`. From Part 3, the two configs that actually converge on the hard matrix:

| config | np=8 iter | np=8 wall | np=1 wall | notes |
|---|---:|---:|---:|---|
| `row_block + pcg + boomeramg` (current default) | 5 ❌ | — | 1.10 s ✅ | works at `np=1`, fails at `np ≥ 3` |
| `metis + gmres + boomeramg` | 1521 | 0.91 s ✅ | 2.39 s ✅ | classical AMG + GMRES, mathematically clean, **uniform across `np`** |
| `row_block + pcg + euclid` | 2103 | **0.36 s ✅** | 0.15 s ❌ subn | **fastest at `np ≥ 2`**, but Euclid+BJ=1 degenerates to a full unstable ILU at `np = 1` |

Two policies on the table:

- **(a) `metis + gmres + boomeramg` everywhere.** One stack for all `np`, robust and "principled" (AMG + GMRES is what Hypre's own docs recommend on hard problems). About 2× slower than baseline at `np = 1` and 1.5× slower than option (b) at `np = 8`. Simplest mental model.

- **(b) Hybrid by `np`.** `np = 1` → `row_block + pcg + boomeramg` (the current baseline, fastest in serial); `np ≥ 2` → `row_block + pcg + euclid` (fastest in parallel). Best per-`np` wall time across the board, but the default depends on `mpi_size` at runtime, which means the wrapper has to switch behavior automatically (a few lines in `HypreSolver`'s constructor).

Trade-off summary:

| policy | np=1 wall | np=8 wall | mental model | implementation cost |
|---|---:|---:|---|---|
| (a) metis + gmres + boomeramg uniform | 2.39 s | 0.91 s | one rule, robust | none (env vars already exposed) |
| (b) hybrid by `np` | **1.10 s** | **0.36 s** | "use Euclid when ≥2 ranks" | ~10 lines: switch precond on `mpi_size_` in the ctor |

**Decision needed: (a), (b), or "leave default as-is and document the override env vars"?**

### Q2. Trilinos: GMRES (current default) or CG?

**Context.** `TrilinosSolver.cpp:290` currently instantiates `Belos::BlockGmresSolMgr` unconditionally. Phase 8 added `POLYSOLVE_TRILINOS_KRYLOV=cg` as an opt-in toggle. On `mat_twist 71_1` step:

| krylov | np=1 wall | np=8 wall |
|---|---:|---:|
| BlockGMRES (current default) | 1.75 s | 0.93 s |
| **PseudoBlockCG** | **0.67 s** | 0.83 s |

CG is faster, especially at `np=1` (~2.6×). For SPD inputs CG is also the textbook choice — the GMRES default looks like a historical defensive setting. **Should we flip the default to CG, or keep GMRES default with CG as opt-in?**

### Q3. Krylov selection policy under "force PSD" — CG-first, GMRES-as-fallback?

**Context.** Our benchmark pipeline runs PolyFEM with `SparseProjectedNewton`, so every matrix in the dataset is SPD by construction (we verified `meta.bin = 1 = projected` across the entire `mat_twist` set). On SPD inputs:

- **CG** is mathematically optimal: minimal memory, fewer flops per iteration, fastest convergence rate.
- **GMRES** still works but is strictly slower and uses much more memory; it pays a robustness price for capabilities (non-SPD tolerance) we don't need.

Should our policy across all solvers be **"use CG by default since we've forced PSD; expose GMRES as an opt-in only for cases where SPD doesn't hold"** (adjoint problems, mixed formulations, raw-Newton variants)? Or keep GMRES default everywhere "just in case"?

This is the meta-question that, once answered, decides Q1 and Q2.